# RQ3 modeling pipeline (age strand + disability strand)

Two modeling tasks per strand:
1. **Overall activity level** -- year x borough x group, predicting the
   inactive/fairly_active/active three-way share 
2. **Activity-specific participation** -- year x borough x group x activity,
   predicting participation for each individual activity, used to answer both
   "will this group participate in this activity" and "which activity has the highest predicted participation rate for this group".



## 0. Setup

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 2026
ALR_EPSILON = 1e-6
AGE_DATA_DIR = Path(r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Shuhan Zhao\q3")
DISABILITY_DATA_DIR = Path(r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\data\processed") 

## 1. Shared functions


In [2]:
def shares_to_alr(shares):
    clipped = np.clip(np.asarray(shares, dtype=float), ALR_EPSILON, 1)
    clipped = clipped / clipped.sum(axis=1, keepdims=True)
    return np.log(clipped[:, :2] / clipped[:, [2]])


def alr_to_shares(values):
    values = np.clip(np.asarray(values, dtype=float), -30, 30)
    exponent = np.exp(values)
    denominator = 1 + exponent.sum(axis=1, keepdims=True)
    return np.column_stack([exponent / denominator, 1 / denominator])


In [3]:
def score_composition(actual, predicted, target_names):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    valid = np.isfinite(actual).all(axis=1) & np.isfinite(predicted).all(axis=1)
    actual, predicted = actual[valid], predicted[valid]
    row = {
        'observations': len(actual),
        'total_variation': np.mean(.5 * np.abs(actual - predicted).sum(axis=1)),
        'overall_mae': np.mean(np.abs(actual - predicted)),
        'overall_rmse': np.sqrt(np.mean((actual - predicted) ** 2)),
    }
    for index, target in enumerate(target_names):
        row[f'{target}_mae'] = mean_absolute_error(actual[:, index], predicted[:, index])
        row[f'{target}_rmse'] = np.sqrt(mean_squared_error(actual[:, index], predicted[:, index]))
        row[f'{target}_r2'] = r2_score(actual[:, index], predicted[:, index])
    return row


def score_single_rate(actual, predicted, target_name):
    actual = np.asarray(actual, dtype=float).ravel()
    predicted = np.asarray(predicted, dtype=float).ravel()
    valid = np.isfinite(actual) & np.isfinite(predicted)
    actual, predicted = actual[valid], predicted[valid]
    return {
        'observations': len(actual),
        f'{target_name}_mae': mean_absolute_error(actual, predicted),
        f'{target_name}_rmse': np.sqrt(mean_squared_error(actual, predicted)),
        f'{target_name}_r2': r2_score(actual, predicted),
    }


In [4]:
def add_lag_features(frame, panel_keys, value_cols, lags=(1,), rolling_window=None, covid_years=(5, 6)):
    prepared = frame.sort_values(panel_keys + ['year']).reset_index(drop=True)
    grouped = prepared.groupby(panel_keys, sort=False)
    for column in value_cols:
        for lag in lags:
            prepared[f'{column}_lag{lag}'] = grouped[column].shift(lag)
        if rolling_window:
            prepared[f'{column}_roll{rolling_window}'] = grouped[column].transform(
                lambda s: s.shift(1).rolling(rolling_window, min_periods=1).mean()
            )
    prepared['time_trend'] = prepared['year'] / prepared['year'].max()
    prepared['is_covid_year'] = prepared['year'].isin(covid_years).astype(int)
    return prepared


def naive_predict(frame, lag_cols):
    return frame[lag_cols].to_numpy()

In [5]:
def add_interaction_terms(frame, group_col, time_col='time_trend'):
    dummies = pd.get_dummies(frame[group_col], prefix=f'{group_col}_x_time')
    interaction_cols = list(dummies.columns)
    frame[interaction_cols] = dummies.mul(frame[time_col], axis=0)
    return frame, interaction_cols

In [6]:
def parameter_candidates(model_name):
    if model_name == 'Ridge Regression':
        return [{'alpha': .1}, {'alpha': 1.0}, {'alpha': 10.0}, {'alpha': 100.0}]
    if model_name == 'Random Forest':
        return [
            {'n_estimators': 160, 'max_depth': 6, 'min_samples_leaf': 5, 'max_features': .5},
            {'n_estimators': 160, 'max_depth': 10, 'min_samples_leaf': 8, 'max_features': .5},
            {'n_estimators': 220, 'max_depth': 8, 'min_samples_leaf': 12, 'max_features': .8},
        ]
    return [
        {'n_estimators': 120, 'learning_rate': .05, 'max_depth': 2, 'min_samples_leaf': 15},
        {'n_estimators': 160, 'learning_rate': .03, 'max_depth': 2, 'min_samples_leaf': 20},
    ]


def build_model(model_name, parameters, numeric_features, categorical_features, multi_output):
    numeric_steps = [('imputer', SimpleImputer(strategy='median'))]
    if model_name == 'Ridge Regression':
        numeric_steps.append(('scale', StandardScaler()))
    preprocess = ColumnTransformer([
        ('numeric', Pipeline(numeric_steps), numeric_features),
        ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ])
    if model_name == 'Ridge Regression':
        estimator = Ridge(**parameters)
    elif model_name == 'Random Forest':
        estimator = RandomForestRegressor(**parameters, random_state=RANDOM_STATE, n_jobs=-1)
    else:
        base = GradientBoostingRegressor(**parameters, random_state=RANDOM_STATE, loss='huber')
        estimator = MultiOutputRegressor(base) if multi_output else base
    return Pipeline([('preprocess', preprocess), ('model', estimator)])


## 2. Task type 1: three-way composition (inactive / fairly_active / active)


In [7]:
def run_composition_task(
    frame,
    panel_keys,
    target_cols,
    weight_col,
    val_year=7,
    test_year=8,
    feature_level='overall'
):
    if feature_level == 'overall':
        prepared = add_lag_features(
            frame,
            panel_keys,
            target_cols,
            lags=(1, 2)
        )
        extra_cols = [
            f'{column}_lag2'
            for column in target_cols
        ]
    else:
        prepared = add_lag_features(
            frame,
            panel_keys,
            target_cols,
            lags=(1,),
            rolling_window=2
        )
        extra_cols = [
            f'{column}_roll2'
            for column in target_cols
        ]

    group_col = panel_keys[1]

    prepared, interaction_cols = add_interaction_terms(
        prepared,
        group_col
    )

    naive_lag_cols = [
        f'{column}_lag1'
        for column in target_cols
    ]

    lag_cols = naive_lag_cols + extra_cols
    categorical = list(panel_keys)

    numeric = (
        lag_cols
        + ['time_trend', 'is_covid_year']
    )

    numeric_by_model = {
        'Ridge Regression': numeric + interaction_cols,
        'Random Forest': numeric,
        'Gradient Boosting': numeric,
    }

    train = (
        prepared[
            prepared['year'] < val_year
        ]
        .dropna(
            subset=target_cols + [weight_col]
        )
        .copy()
    )

    val = (
        prepared[
            prepared['year'] == val_year
        ]
        .dropna(
            subset=target_cols + lag_cols
        )
        .copy()
    )

    trainval = (
        prepared[
            prepared['year'] < test_year
        ]
        .dropna(
            subset=target_cols + [weight_col]
        )
        .copy()
    )

    test = (
        prepared[
            prepared['year'] == test_year
        ]
        .dropna(
            subset=target_cols + lag_cols
        )
        .copy()
    )

    results = {}

    for label, split in [
        ('validation', val),
        ('test', test)
    ]:
        naive_prediction = naive_predict(
            split,
            naive_lag_cols
        )

        results[
            ('Naive baseline', label)
        ] = score_composition(
            split[target_cols],
            naive_prediction,
            target_cols
        )

    model_names = [
        'Ridge Regression',
        'Random Forest',
        'Gradient Boosting'
    ]

    for model_name in model_names:
        numeric_for_model = numeric_by_model[
            model_name
        ]

        best_params = None
        best_score = np.inf
        best_validation_metrics = None

        for params in parameter_candidates(model_name):
            model = build_model(
                model_name,
                params,
                numeric_for_model,
                categorical,
                multi_output=True
            )

            train_weights = (
                train[weight_col]
                .to_numpy()
            )

            model.fit(
                train[
                    numeric_for_model + categorical
                ],
                shares_to_alr(
                    train[target_cols]
                ),
                model__sample_weight=train_weights
            )

            validation_prediction = alr_to_shares(
                model.predict(
                    val[
                        numeric_for_model
                        + categorical
                    ]
                )
            )

            validation_metrics = score_composition(
                val[target_cols],
                validation_prediction,
                target_cols
            )

            validation_score = (
                validation_metrics['overall_mae']
            )

            if validation_score < best_score:
                best_score = validation_score
                best_params = params
                best_validation_metrics = (
                    validation_metrics
                )

        results[
            (model_name, 'validation')
        ] = best_validation_metrics

        final_model = build_model(
            model_name,
            best_params,
            numeric_for_model,
            categorical,
            multi_output=True
        )

        trainval_weights = (
            trainval[weight_col]
            .to_numpy()
        )

        final_model.fit(
            trainval[
                numeric_for_model + categorical
            ],
            shares_to_alr(
                trainval[target_cols]
            ),
            model__sample_weight=trainval_weights
        )

        test_prediction = alr_to_shares(
            final_model.predict(
                test[
                    numeric_for_model
                    + categorical
                ]
            )
        )

        results[
            (model_name, 'test')
        ] = score_composition(
            test[target_cols],
            test_prediction,
            target_cols
        )

        results[
            (model_name, 'best_params')
        ] = best_params

        results[
            (model_name, 'fitted_model')
        ] = final_model

    return results, prepared

### 2.1 Age strand -- overall activity level

In [8]:
age_overall = pd.read_csv(AGE_DATA_DIR / 'q3_age_overall_activity_level_panel.csv')
age_overall['LA_2023'] = age_overall['LA_2023'].astype('Int64').astype(str)

age_overall_targets = ['overall_inactive_rate', 'overall_fairly_active_rate', 'overall_active_rate']
age_overall_results, age_overall_panel = run_composition_task(
    age_overall,
    panel_keys=['LA_2023', 'age_group'],
    target_cols=age_overall_targets,
    weight_col='weighted_n_overall_activity_level',
    feature_level='overall',
)

for key, value in age_overall_results.items():
    if key[1] == 'test':
        print(key, {k: (round(v, 4) if isinstance(v, float) else v) for k, v in value.items()})

('Naive baseline', 'test') {'observations': 256, 'total_variation': np.float64(0.1377), 'overall_mae': np.float64(0.0918), 'overall_rmse': np.float64(0.1432), 'overall_inactive_rate_mae': 0.1034, 'overall_inactive_rate_rmse': np.float64(0.162), 'overall_inactive_rate_r2': 0.0572, 'overall_fairly_active_rate_mae': 0.0616, 'overall_fairly_active_rate_rmse': np.float64(0.085), 'overall_fairly_active_rate_r2': -1.0996, 'overall_active_rate_mae': 0.1104, 'overall_active_rate_rmse': np.float64(0.1674), 'overall_active_rate_r2': 0.0992}
('Ridge Regression', 'test') {'observations': 256, 'total_variation': np.float64(0.1221), 'overall_mae': np.float64(0.0814), 'overall_rmse': np.float64(0.1366), 'overall_inactive_rate_mae': 0.0983, 'overall_inactive_rate_rmse': np.float64(0.1666), 'overall_inactive_rate_r2': 0.0026, 'overall_fairly_active_rate_mae': 0.0508, 'overall_fairly_active_rate_rmse': np.float64(0.0754), 'overall_fairly_active_rate_r2': -0.649, 'overall_active_rate_mae': 0.0952, 'overal

### 2.2 Disability strand -- overall activity level



In [9]:
dis_overall = pd.read_csv(DISABILITY_DATA_DIR / 'RQ3_borough_disability_overall_all_years.csv')
dis_overall['LA_2023'] = dis_overall['LA_2023'].astype('Int64').astype(str)

dis_overall_targets = ['inactive_rate', 'fairly_active_rate', 'active_rate']
dis_overall_results, dis_overall_panel = run_composition_task(
    dis_overall,
    panel_keys=['LA_2023', 'disability_group'],
    target_cols=dis_overall_targets,
    weight_col='weighted_n',
    feature_level='overall',
)

for key, value in dis_overall_results.items():
    if key[1] == 'test':
        print(key, {k: (round(v, 4) if isinstance(v, float) else v) for k, v in value.items()})

('Naive baseline', 'test') {'observations': 504, 'total_variation': np.float64(0.2494), 'overall_mae': np.float64(0.1662), 'overall_rmse': np.float64(0.2454), 'inactive_rate_mae': 0.1934, 'inactive_rate_rmse': np.float64(0.2674), 'inactive_rate_r2': -0.7096, 'fairly_active_rate_mae': 0.0998, 'fairly_active_rate_rmse': np.float64(0.1608), 'fairly_active_rate_r2': -1.1113, 'active_rate_mae': 0.2055, 'active_rate_rmse': np.float64(0.2887), 'active_rate_r2': -0.853}
('Ridge Regression', 'test') {'observations': 504, 'total_variation': np.float64(0.2079), 'overall_mae': np.float64(0.1386), 'overall_rmse': np.float64(0.2029), 'inactive_rate_mae': 0.1564, 'inactive_rate_rmse': np.float64(0.2132), 'inactive_rate_r2': -0.0867, 'fairly_active_rate_mae': 0.0797, 'fairly_active_rate_rmse': np.float64(0.1359), 'fairly_active_rate_r2': -0.5084, 'active_rate_mae': 0.1797, 'active_rate_rmse': np.float64(0.2442), 'active_rate_r2': -0.3258}
('Random Forest', 'test') {'observations': 504, 'total_variatio

### 2.3 Disability strand -- activity-specific MEMS7GR tiers



In [10]:
dis_level = pd.read_csv(DISABILITY_DATA_DIR / 'RQ3_borough_disability_MEMS7GR_all_years.csv')
dis_level['LA_2023'] = dis_level['LA_2023'].astype('Int64').astype(str)

dis_level_targets = ['inactive_rate', 'fairly_active_rate', 'active_rate']
dis_level_results, dis_level_panel = run_composition_task(
    dis_level,
    panel_keys=['LA_2023', 'disability_group', 'activity'],
    target_cols=dis_level_targets,
    weight_col='weighted_n',
    feature_level='activity',
)

for key, value in dis_level_results.items():
    if key[1] == 'test':
        print(key, {k: (round(v, 4) if isinstance(v, float) else v) for k, v in value.items()})

('Naive baseline', 'test') {'observations': 62392, 'total_variation': np.float64(0.0114), 'overall_mae': np.float64(0.0076), 'overall_rmse': np.float64(0.0442), 'inactive_rate_mae': 0.0103, 'inactive_rate_rmse': np.float64(0.0529), 'inactive_rate_r2': 0.1265, 'fairly_active_rate_mae': 0.0056, 'fairly_active_rate_rmse': np.float64(0.0351), 'fairly_active_rate_r2': -0.6335, 'active_rate_mae': 0.0069, 'active_rate_rmse': np.float64(0.0427), 'active_rate_r2': -0.0694}
('Ridge Regression', 'test') {'observations': 62392, 'total_variation': np.float64(0.0092), 'overall_mae': np.float64(0.0061), 'overall_rmse': np.float64(0.0381), 'inactive_rate_mae': 0.0089, 'inactive_rate_rmse': np.float64(0.0486), 'inactive_rate_r2': 0.2618, 'fairly_active_rate_mae': 0.004, 'fairly_active_rate_rmse': np.float64(0.0268), 'fairly_active_rate_r2': 0.0481, 'active_rate_mae': 0.0053, 'active_rate_rmse': np.float64(0.0355), 'active_rate_r2': 0.2596}
('Random Forest', 'test') {'observations': 62392, 'total_variat

## 3. Task type 2: single participation rate (MONTHS_12 / DAYS10P60GR)



In [11]:
def run_single_rate_task(
    frame,
    panel_keys,
    target_col,
    weight_col,
    val_year=7,
    test_year=8,
    feature_level='activity'
):
    if feature_level == 'overall':
        prepared = add_lag_features(
            frame,
            panel_keys,
            [target_col],
            lags=(1, 2)
        )

        extra_cols = [
            f'{target_col}_lag2'
        ]
    else:
        prepared = add_lag_features(
            frame,
            panel_keys,
            [target_col],
            lags=(1,),
            rolling_window=2
        )

        extra_cols = [
            f'{target_col}_roll2'
        ]

    group_col = panel_keys[1]

    prepared, interaction_cols = add_interaction_terms(
        prepared,
        group_col
    )

    lag_col = f'{target_col}_lag1'
    lag_cols = [lag_col] + extra_cols
    categorical = list(panel_keys)

    numeric = (
        lag_cols
        + ['time_trend', 'is_covid_year']
    )

    numeric_by_model = {
        'Ridge Regression': numeric + interaction_cols,
        'Random Forest': numeric,
        'Gradient Boosting': numeric,
    }

    train = (
        prepared[
            prepared['year'] < val_year
        ]
        .dropna(
            subset=[target_col, weight_col]
        )
        .copy()
    )

    val = (
        prepared[
            prepared['year'] == val_year
        ]
        .dropna(
            subset=[target_col] + lag_cols
        )
        .copy()
    )

    trainval = (
        prepared[
            prepared['year'] < test_year
        ]
        .dropna(
            subset=[target_col, weight_col]
        )
        .copy()
    )

    test = (
        prepared[
            prepared['year'] == test_year
        ]
        .dropna(
            subset=[target_col] + lag_cols
        )
        .copy()
    )

    results = {}

    for label, split in [
        ('validation', val),
        ('test', test)
    ]:
        naive_prediction = naive_predict(
            split,
            [lag_col]
        )

        results[
            ('Naive baseline', label)
        ] = score_single_rate(
            split[target_col],
            naive_prediction,
            target_col
        )

    model_names = [
        'Ridge Regression',
        'Random Forest',
        'Gradient Boosting'
    ]

    for model_name in model_names:
        numeric_for_model = numeric_by_model[
            model_name
        ]

        best_params = None
        best_score = np.inf
        best_validation_metrics = None

        for params in parameter_candidates(model_name):
            model = build_model(
                model_name,
                params,
                numeric_for_model,
                categorical,
                multi_output=False
            )

            train_weights = (
                train[weight_col]
                .to_numpy()
            )

            model.fit(
                train[
                    numeric_for_model + categorical
                ],
                train[target_col],
                model__sample_weight=train_weights
            )

            validation_prediction = np.clip(
                model.predict(
                    val[
                        numeric_for_model
                        + categorical
                    ]
                ),
                0,
                1
            )

            validation_metrics = score_single_rate(
                val[target_col],
                validation_prediction,
                target_col
            )

            validation_score = validation_metrics[
                f'{target_col}_mae'
            ]

            if validation_score < best_score:
                best_score = validation_score
                best_params = params
                best_validation_metrics = (
                    validation_metrics
                )

        results[
            (model_name, 'validation')
        ] = best_validation_metrics

        final_model = build_model(
            model_name,
            best_params,
            numeric_for_model,
            categorical,
            multi_output=False
        )

        trainval_weights = (
            trainval[weight_col]
            .to_numpy()
        )

        final_model.fit(
            trainval[
                numeric_for_model + categorical
            ],
            trainval[target_col],
            model__sample_weight=trainval_weights
        )

        test_prediction = np.clip(
            final_model.predict(
                test[
                    numeric_for_model
                    + categorical
                ]
            ),
            0,
            1
        )

        results[
            (model_name, 'test')
        ] = score_single_rate(
            test[target_col],
            test_prediction,
            target_col
        )

        results[
            (model_name, 'best_params')
        ] = best_params

        results[
            (model_name, 'fitted_model')
        ] = final_model

    return results, prepared, test

### 3.1 Disability strand -- MONTHS_12 and DAYS10P60GR



In [12]:
dis_dm = pd.read_csv(DISABILITY_DATA_DIR / 'RQ3_borough_disability_days_months_all_years.csv')
dis_dm['LA_2023'] = dis_dm['LA_2023'].astype('Int64').astype(str)

months12_results, months12_panel, months12_test = run_single_rate_task(
    dis_dm, ['LA_2023', 'disability_group', 'activity'],
    target_col='participation_MONTHS_12', weight_col='weighted_n_MONTHS_12',
    feature_level='activity',
)
days_results, days_panel, days_test = run_single_rate_task(
    dis_dm, ['LA_2023', 'disability_group', 'activity'],
    target_col='participation_DAYS10P60GR', weight_col='weighted_n_DAYS10P60GR',
    feature_level='activity',
)

for label, results in [('MONTHS_12', months12_results), ('DAYS10P60GR', days_results)]:
    print(f'--- {label} ---')
    for key, value in results.items():
        if key[1] == 'test':
            print(key, {k: (round(v, 4) if isinstance(v, float) else v) for k, v in value.items()})

--- MONTHS_12 ---
('Naive baseline', 'test') {'observations': 62392, 'participation_MONTHS_12_mae': 0.0224, 'participation_MONTHS_12_rmse': np.float64(0.0788), 'participation_MONTHS_12_r2': 0.3746}
('Ridge Regression', 'test') {'observations': 62392, 'participation_MONTHS_12_mae': 0.0213, 'participation_MONTHS_12_rmse': np.float64(0.061), 'participation_MONTHS_12_r2': 0.6257}
('Random Forest', 'test') {'observations': 62392, 'participation_MONTHS_12_mae': 0.0228, 'participation_MONTHS_12_rmse': np.float64(0.0586), 'participation_MONTHS_12_r2': 0.6539}
('Gradient Boosting', 'test') {'observations': 62392, 'participation_MONTHS_12_mae': 0.0212, 'participation_MONTHS_12_rmse': np.float64(0.0645), 'participation_MONTHS_12_r2': 0.5813}
--- DAYS10P60GR ---
('Naive baseline', 'test') {'observations': 62392, 'participation_DAYS10P60GR_mae': 0.0109, 'participation_DAYS10P60GR_rmse': np.float64(0.0547), 'participation_DAYS10P60GR_r2': 0.2277}
('Ridge Regression', 'test') {'observations': 62392, 

### 3.2 Age strand -- months12_rate and days10p60gr_rate from the complete table



In [13]:
age_complete = pd.read_csv(AGE_DATA_DIR / 'q3_age_activity_participation_level_panel_complete.csv')
age_complete['LA_2023'] = age_complete['LA_2023'].astype('Int64').astype(str)

age_months12_results, age_months12_panel, age_months12_test = run_single_rate_task(
    age_complete, ['LA_2023', 'age_group', 'activity_suffix'],
    target_col='months12_rate', weight_col='weighted_n_months12',
    feature_level='activity',
)
age_days_results, age_days_panel, age_days_test = run_single_rate_task(
    age_complete, ['LA_2023', 'age_group', 'activity_suffix'],
    target_col='days10p60gr_rate', weight_col='weighted_n_days10p60gr',
    feature_level='activity',
)

age_complete_targets = ['activity_inactive_rate', 'activity_fairly_active_rate', 'activity_active_rate']
age_level_results, age_level_panel = run_composition_task(
    age_complete, panel_keys=['LA_2023', 'age_group', 'activity_suffix'],
    target_cols=age_complete_targets, weight_col='weighted_n_activity_level',
    feature_level='activity',
)

## 4. Extracting the highest predicted participation activity

In [14]:
def select_top_activity(
    working,
    id_cols,
    activity_col,
    prediction_col,
    top_n=1
):
    ranked = working.sort_values(
        id_cols + [prediction_col],
        ascending=(
            [True] * len(id_cols)
            + [False]
        )
    )

    return (
        ranked
        .groupby(
            id_cols,
            as_index=False
        )
        .head(top_n)[
            id_cols
            + [activity_col, prediction_col]
        ]
    )


validation_metric = (
    'participation_MONTHS_12_mae'
)

candidate_names = [
    'Naive baseline',
    'Ridge Regression',
    'Random Forest',
    'Gradient Boosting'
]

validation_mae = {
    model_name: months12_results[
        (model_name, 'validation')
    ][validation_metric]
    for model_name in candidate_names
}

best_disability_model_name = min(
    validation_mae,
    key=validation_mae.get
)

print(
    'Validation MAE by method:'
)

for model_name, mae_value in validation_mae.items():
    print(
        model_name,
        round(mae_value, 6)
    )

print(
    'Selected method for highest predicted participation activity:',
    best_disability_model_name
)

base_numeric_cols = [
    'participation_MONTHS_12_lag1',
    'participation_MONTHS_12_roll2',
    'time_trend',
    'is_covid_year'
]

categorical_cols = [
    'LA_2023',
    'disability_group',
    'activity'
]

if (
    best_disability_model_name
    == 'Naive baseline'
):
    preferred_working = (
        months12_test
        .dropna(
            subset=(
                categorical_cols
                + [
                    'participation_MONTHS_12_lag1'
                ]
            )
        )
        .copy()
    )

    preferred_working[
        'predicted_participation'
    ] = np.clip(
        preferred_working[
            'participation_MONTHS_12_lag1'
        ],
        0,
        1
    )

else:
    best_disability_model = months12_results[
        (
            best_disability_model_name,
            'fitted_model'
        )
    ]

    numeric_cols = list(base_numeric_cols)

    if (
        best_disability_model_name
        == 'Ridge Regression'
    ):
        interaction_cols = [
            column
            for column in months12_test.columns
            if column.startswith(
                'disability_group_x_time_'
            )
        ]

        numeric_cols = (
            numeric_cols
            + interaction_cols
        )

    preferred_working = (
        months12_test
        .dropna(
            subset=(
                numeric_cols
                + categorical_cols
            )
        )
        .copy()
    )

    preferred_working[
        'predicted_participation'
    ] = np.clip(
        best_disability_model.predict(
            preferred_working[
                numeric_cols
                + categorical_cols
            ]
        ),
        0,
        1
    )

top_activity_by_disability_group = (
    select_top_activity(
        preferred_working,
        id_cols=[
            'LA_2023',
            'disability_group'
        ],
        activity_col='activity',
        prediction_col=(
            'predicted_participation'
        ),
        top_n=1
    )
)

top_activity_by_disability_group.head(10)

Validation MAE by method:
Naive baseline 0.020662
Ridge Regression 0.020366
Random Forest 0.022582
Gradient Boosting 0.019796
Selected method for highest predicted participation activity: Gradient Boosting


,LA_2023,disability_group,activity,predicted_participation
15,107,disty1,ACTTRAV_C03,0.502279
1007,107,disty10,ACTTRAV_C03,0.363598
1999,107,disty11,ACTTRAV_C03,0.337847
2991,107,disty12,ACTTRAV_C03,0.370122
3983,107,disty13,ACTTRAV_C03,0.402565
4975,107,disty2,ACTTRAV_C03,0.466040
5967,107,disty3,ACTTRAV_C03,0.466040
6959,107,disty4,ACTTRAV_C03,0.543043
7951,107,disty5,ACTTRAV_C03,0.584399
8943,107,disty6,ACTTRAV_C03,0.308201


## 5. Summary table



In [15]:
def collect_summary(results_dict, task_name):
    rows = []

    for (model_name, split), value in results_dict.items():
        if (
            split in ['validation', 'test']
            and isinstance(value, dict)
        ):
            rows.append({
                'task': task_name,
                'model': model_name,
                'split': split,
                **value
            })

    return pd.DataFrame(rows)


summary = pd.concat(
    [
        collect_summary(
            age_overall_results,
            'age_overall_level'
        ),
        collect_summary(
            dis_overall_results,
            'disability_overall_level'
        ),
        collect_summary(
            dis_level_results,
            'disability_activity_level'
        ),
        collect_summary(
            months12_results,
            'disability_months12'
        ),
        collect_summary(
            days_results,
            'disability_days10p60gr'
        ),
        collect_summary(
            age_months12_results,
            'age_months12'
        ),
        collect_summary(
            age_days_results,
            'age_days10p60gr'
        ),
        collect_summary(
            age_level_results,
            'age_activity_level'
        ),
    ],
    ignore_index=True
)

summary = summary.sort_values(
    ['task', 'split', 'model']
).reset_index(drop=True)

print(
    summary[
        ['task', 'model', 'split', 'observations']
    ]
)

summary

                        task              model       split  observations
0         age_activity_level  Gradient Boosting        test         31585
1         age_activity_level     Naive baseline        test         31585
2         age_activity_level      Random Forest        test         31585
3         age_activity_level   Ridge Regression        test         31585
4         age_activity_level  Gradient Boosting  validation         31638
..                       ...                ...         ...           ...
59  disability_overall_level   Ridge Regression        test           504
60  disability_overall_level  Gradient Boosting  validation           497
61  disability_overall_level     Naive baseline  validation           497
62  disability_overall_level      Random Forest  validation           497
63  disability_overall_level   Ridge Regression  validation           497

[64 rows x 4 columns]


,task,model,split,observations,total_variation,overall_mae,overall_rmse,overall_inactive_rate_mae,overall_inactive_rate_rmse,overall_inactive_rate_r2,...,days10p60gr_rate_r2,activity_inactive_rate_mae,activity_inactive_rate_rmse,activity_inactive_rate_r2,activity_fairly_active_rate_mae,activity_fairly_active_rate_rmse,activity_fairly_active_rate_r2,activity_active_rate_mae,activity_active_rate_rmse,activity_active_rate_r2
0,age_activity_level,Gradient Boosting,test,31585,0.008343,0.005562,0.024559,NaN,NaN,NaN,...,NaN,0.007781,0.030364,0.731242,0.003791,0.016729,0.499745,0.005114,0.024649,0.635443
1,age_activity_level,Naive baseline,test,31585,0.009492,0.006328,0.025180,NaN,NaN,NaN,...,NaN,0.008348,0.030253,0.733191,0.004883,0.020013,0.284075,0.005754,0.024212,0.648236
2,age_activity_level,Random Forest,test,31585,0.008033,0.005355,0.024109,NaN,NaN,NaN,...,NaN,0.007687,0.030736,0.724604,0.003797,0.016966,0.485503,0.004581,0.022610,0.693254
3,age_activity_level,Ridge Regression,test,31585,0.008937,0.005958,0.026516,NaN,NaN,NaN,...,NaN,0.008651,0.034337,0.656292,0.004214,0.018378,0.396276,0.005009,0.024342,0.644456
4,age_activity_level,Gradient Boosting,validation,31638,0.007999,0.005333,0.023220,NaN,NaN,NaN,...,NaN,0.007449,0.028948,0.732908,0.003629,0.016100,0.513617,0.004919,0.022812,0.644652
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,disability_overall_level,Ridge Regression,test,504,0.207858,0.138572,0.202944,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
60,disability_overall_level,Gradient Boosting,validation,497,0.183378,0.122252,0.181070,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
61,disability_overall_level,Naive baseline,validation,497,0.257090,0.171393,0.257571,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
62,disability_overall_level,Random Forest,validation,497,0.189476,0.126317,0.190119,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
import os
import pickle

output_dir = r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\outputs"
os.makedirs(output_dir, exist_ok=True)

# 1. summary table
summary.to_csv(os.path.join(output_dir, 'summary_full.csv'), index=False)

# 2. highest predicted participation activity table
top_activity_by_disability_group.to_csv(
    os.path.join(output_dir, 'top_activity_by_disability_group.csv'), index=False
)

# 3. every fitted model from every task
all_results = {
    'age_overall': age_overall_results,
    'dis_overall': dis_overall_results,
    'dis_level': dis_level_results,
    'months12': months12_results,
    'days': days_results,
    'age_months12': age_months12_results,
    'age_days': age_days_results,
    'age_level': age_level_results,
}

fitted_models = {}
for task_name, results in all_results.items():
    for (model_name, key), value in results.items():
        if key == 'fitted_model':
            fitted_models[f'{task_name}__{model_name}'] = value

with open(os.path.join(output_dir, 'fitted_models.pkl'), 'wb') as f:
    pickle.dump(fitted_models, f)

# 4. best_params for every task/model
best_params_rows = []
for task_name, results in all_results.items():
    for (model_name, key), value in results.items():
        if key == 'best_params':
            best_params_rows.append({'task': task_name, 'model': model_name, **value})

import pandas as pd
pd.DataFrame(best_params_rows).to_csv(
    os.path.join(output_dir, 'best_hyperparameters.csv'), index=False
)

print('Saved to', output_dir)
print('- summary_full.csv')
print('- top_activity_by_disability_group.csv')
print('- fitted_models.pkl', f'({len(fitted_models)} models)')
print('- best_hyperparameters.csv')

Saved to C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\outputs
- summary_full.csv
- top_activity_by_disability_group.csv
- fitted_models.pkl (24 models)
- best_hyperparameters.csv


## 6. Activity coverage check

In [17]:
print("MEMS7GR activities:", dis_level["activity"].nunique())
print("DAYS10P60GR and MONTHS_12 activities:", dis_dm["activity"].nunique())

assert dis_level["activity"].nunique() == 124
assert dis_dm["activity"].nunique() == 124
assert "HULAHOOP_P27" not in set(dis_level["activity"])
assert "HULAHOOP_P27" not in set(dis_dm["activity"])

MEMS7GR activities: 124
DAYS10P60GR and MONTHS_12 activities: 124
